In [22]:
import pandas as pd
import numpy as np
import optuna

from nltk.corpus import stopwords

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_DIR = "../data"

PREPS = ["normal", "stem", "lemma"]
VARIANTES = ["mx", "es", "cu"]

FEATURE_COLS = [
    "n_exc", "n_int", "n_may",
    "n_emo", "n_ris", "n_neg",
    "n_elo", "n_com", "n_pun"
]

PREP_GANADOR = {
    "mx": "lemma",
    "es": "normal",
    "cu": "normal"
}

STOP_WORDS = stopwords.words("spanish")

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [23]:
def cargar_train_ganador(variante):
    prep = PREP_GANADOR[variante]

    sufijo = "" if prep == "normal" else f"_{prep}"

    path = f"{DATA_DIR}/train_clean{sufijo}_{variante}.csv"

    df = pd.read_csv(path)

    return df

In [24]:
def build_tfidf_prep(ngram_range=(1, 2)):

    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=ngram_range,
        min_df=2,
        max_df=0.98,
        max_features=20000,
        lowercase=False,
        sublinear_tf=True,
        dtype=np.float32
    )

    return ColumnTransformer([
        (
            "tfidf",
            tfidf,
            "MESSAGE_CLEAN"
        ),
        (
            "ling",
            "passthrough",
            FEATURE_COLS
        )
    ])

In [25]:
def objective(trial, df_train):

    X = df_train[
        ["MESSAGE_CLEAN"] + FEATURE_COLS
    ]

    y = df_train["IS_IRONIC"].values

    scale_pos_weight = (
        (y == 0).sum() /
        (y == 1).sum()
    )

    ngram_range = trial.suggest_categorical(
        "ngram_range",
        ["1_1", "1_2"]
    )

    NGRAM_MAP = {
        "1_1": (1, 1),
        "1_2": (1, 2)
    }

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 600, step=50
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
            log=True
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            5.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10.0,
            log=True
        )
    }

    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        (
            "prep",
            build_tfidf_prep(
                NGRAM_MAP[ngram_range]
            )
        ),
        ("clf", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="f1_macro",
        n_jobs=1
    )

    return scores.mean()

In [26]:
def optimizar_variante(variante, n_trials=50):

    df_train = cargar_train_ganador(variante)

    print(
        f"\nOptimización XGBoost | "
        f"{variante.upper()} | "
        f"prep={PREP_GANADOR[variante]}"
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        )
    )

    study.optimize(
        lambda trial: objective(
            trial,
            df_train
        ),
        n_trials=n_trials
    )

    print("\nMejor F1-Macro CV:")
    print(study.best_value)

    print("\nMejores hiperparámetros:")
    print(study.best_params)

    return study

In [27]:
study_mx = optimizar_variante("mx", n_trials=50)
study_es = optimizar_variante("es", n_trials=50)
study_cu = optimizar_variante("cu", n_trials=50)

[I 2026-09-21 15:58:20,330] A new study created in memory with name: no-name-963396a1-3c40-44d0-8256-cf68697039d6



Optimización XGBoost | MX | prep=lemma


[I 2026-09-21 15:58:27,245] Trial 0 finished with value: 0.611432874222849 and parameters: {'ngram_range': '1_2', 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01700037298921102, 'min_child_weight': 2, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.9330880728874675, 'gamma': 3.005575058716044, 'reg_alpha': 0.21242802137208885, 'reg_lambda': 0.10994335574766201}. Best is trial 0 with value: 0.611432874222849.
[I 2026-09-21 15:58:28,467] Trial 1 finished with value: 0.5869644192508019 and parameters: {'ngram_range': '1_1', 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.018659959624904916, 'min_child_weight': 4, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7159725093210578, 'gamma': 1.4561457009902097, 'reg_alpha': 0.07500295933641415, 'reg_lambda': 0.19010245319870356}. Best is trial 0 with value: 0.611432874222849.
[I 2026-09-21 15:58:33,120] Trial 2 finished with value: 0.6022000356498525 and parameters: {'ngram_range': '1_2', 'n_estimators': 350, '


Mejor F1-Macro CV:
0.6235171902192473

Mejores hiperparámetros:
{'ngram_range': '1_2', 'n_estimators': 550, 'max_depth': 7, 'learning_rate': 0.01659694548952407, 'min_child_weight': 3, 'subsample': 0.7375238849252976, 'colsample_bytree': 0.6616935367367804, 'gamma': 3.5691762015269073, 'reg_alpha': 0.0004084687876720793, 'reg_lambda': 0.5745153746962762}

Optimización XGBoost | ES | prep=normal


[I 2026-09-21 16:03:06,409] Trial 0 finished with value: 0.6950482057849255 and parameters: {'ngram_range': '1_2', 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01700037298921102, 'min_child_weight': 2, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.9330880728874675, 'gamma': 3.005575058716044, 'reg_alpha': 0.21242802137208885, 'reg_lambda': 0.10994335574766201}. Best is trial 0 with value: 0.6950482057849255.
[I 2026-09-21 16:03:07,600] Trial 1 finished with value: 0.6770818539665628 and parameters: {'ngram_range': '1_1', 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.018659959624904916, 'min_child_weight': 4, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7159725093210578, 'gamma': 1.4561457009902097, 'reg_alpha': 0.07500295933641415, 'reg_lambda': 0.19010245319870356}. Best is trial 0 with value: 0.6950482057849255.
[I 2026-09-21 16:03:14,841] Trial 2 finished with value: 0.6911593593657821 and parameters: {'ngram_range': '1_2', 'n_estimators': 350


Mejor F1-Macro CV:
0.703953521389756

Mejores hiperparámetros:
{'ngram_range': '1_1', 'n_estimators': 550, 'max_depth': 6, 'learning_rate': 0.018576966954164022, 'min_child_weight': 1, 'subsample': 0.6402182019004918, 'colsample_bytree': 0.8953495839474268, 'gamma': 1.9385073783826383, 'reg_alpha': 0.14534984477447524, 'reg_lambda': 0.11013004277239107}

Optimización XGBoost | CU | prep=normal


[I 2026-09-21 16:07:48,340] Trial 0 finished with value: 0.6561369545795308 and parameters: {'ngram_range': '1_2', 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01700037298921102, 'min_child_weight': 2, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.9330880728874675, 'gamma': 3.005575058716044, 'reg_alpha': 0.21242802137208885, 'reg_lambda': 0.10994335574766201}. Best is trial 0 with value: 0.6561369545795308.
[I 2026-09-21 16:07:49,616] Trial 1 finished with value: 0.6509671728456838 and parameters: {'ngram_range': '1_1', 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.018659959624904916, 'min_child_weight': 4, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7159725093210578, 'gamma': 1.4561457009902097, 'reg_alpha': 0.07500295933641415, 'reg_lambda': 0.19010245319870356}. Best is trial 0 with value: 0.6561369545795308.
[I 2026-09-21 16:07:56,497] Trial 2 finished with value: 0.6528777972774844 and parameters: {'ngram_range': '1_2', 'n_estimators': 350


Mejor F1-Macro CV:
0.6665622610215698

Mejores hiperparámetros:
{'ngram_range': '1_1', 'n_estimators': 600, 'max_depth': 4, 'learning_rate': 0.021187376439248576, 'min_child_weight': 2, 'subsample': 0.679064775548546, 'colsample_bytree': 0.9984406625629536, 'gamma': 1.5234621896540699, 'reg_alpha': 0.0052269265404755955, 'reg_lambda': 0.15877742777317128}


In [28]:
resumen_optuna = pd.DataFrame([
    {
        "variante": "mx",
        "preprocessing": PREP_GANADOR["mx"],
        "best_f1_macro_cv": study_mx.best_value,
        **study_mx.best_params
    },
    {
        "variante": "es",
        "preprocessing": PREP_GANADOR["es"],
        "best_f1_macro_cv": study_es.best_value,
        **study_es.best_params
    },
    {
        "variante": "cu",
        "preprocessing": PREP_GANADOR["cu"],
        "best_f1_macro_cv": study_cu.best_value,
        **study_cu.best_params
    }
])

resumen_optuna

,variante,preprocessing,best_f1_macro_cv,ngram_range,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,mx,lemma,0.623517,1_2,550,7,0.016597,3,0.737524,0.661694,3.569176,0.000408,0.574515
1,es,normal,0.703954,1_1,550,6,0.018577,1,0.640218,0.895350,1.938507,0.145350,0.110130
2,cu,normal,0.666562,1_1,600,4,0.021187,2,0.679065,0.998441,1.523462,0.005227,0.158777


In [29]:
f1_base = {
    "mx": 0.600949,
    "es": 0.686399,
    "cu": 0.653014
}

comparacion = resumen_optuna[
    ["variante", "preprocessing", "best_f1_macro_cv", "ngram_range"]
].copy()

comparacion["f1_xgb_base"] = comparacion["variante"].map(f1_base)

comparacion["mejora_optuna"] = (
    comparacion["best_f1_macro_cv"]
    - comparacion["f1_xgb_base"]
)

comparacion = comparacion[
    [
        "variante",
        "preprocessing",
        "f1_xgb_base",
        "best_f1_macro_cv",
        "mejora_optuna",
        "ngram_range"
    ]
]

comparacion

,variante,preprocessing,f1_xgb_base,best_f1_macro_cv,mejora_optuna,ngram_range
0,mx,lemma,0.600949,0.623517,0.022568,1_2
1,es,normal,0.686399,0.703954,0.017555,1_1
2,cu,normal,0.653014,0.666562,0.013548,1_1


In [30]:
def objective_final(trial, df_train):

    X = df_train[
        ["MESSAGE_CLEAN"] + FEATURE_COLS
    ]

    y = df_train["IS_IRONIC"].values

    scale_pos_weight = (
        (y == 0).sum() /
        (y == 1).sum()
    )

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 600, step=50
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
            log=True
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.4, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            5.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.01,
            10.0,
            log=True
        )
    }

    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline([
        (
            "prep",
            build_tfidf_prep((1, 2))
        ),
        ("clf", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="f1_macro",
        n_jobs=1
    )

    return scores.mean()

In [31]:
def optimizar_variante_final(variante, n_trials=50):

    df_train = cargar_train_ganador(variante)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        )
    )

    study.optimize(
        lambda trial: objective_final(
            trial,
            df_train
        ),
        n_trials=n_trials
    )

    print(
        f"{variante.upper()} | "
        f"F1-Macro={study.best_value:.6f}"
    )

    print(study.best_params)

    return study

In [32]:
study_mx_final = optimizar_variante_final(
    "mx",
    n_trials=50
)

study_es_final = optimizar_variante_final(
    "es",
    n_trials=50
)

study_cu_final = optimizar_variante_final(
    "cu",
    n_trials=50
)

[I 2026-09-21 16:12:17,016] A new study created in memory with name: no-name-1bcaf185-9800-4c88-9567-66713a5f0f5b
[I 2026-09-21 16:12:21,338] Trial 0 finished with value: 0.5898527423946123 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.49359671220172163, 'gamma': 0.2904180608409973, 'reg_alpha': 1.1752647960576206, 'reg_lambda': 0.6358358856676253}. Best is trial 0 with value: 0.5898527423946123.
[I 2026-09-21 16:12:24,152] Trial 1 finished with value: 0.5863772669119584 and parameters: {'n_estimators': 450, 'max_depth': 2, 'learning_rate': 0.2708160864249968, 'min_child_weight': 9, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'gamma': 0.9170225492671691, 'reg_alpha': 0.0026892128247368906, 'reg_lambda': 0.3752055855124282}. Best is trial 0 with value: 0.5898527423946123.
[I 2026-09-21 16:12:27,070] Trial 2 finished with value: 0.603762

MX | F1-Macro=0.625833
{'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.06299349502760707, 'min_child_weight': 3, 'subsample': 0.7683638216913562, 'colsample_bytree': 0.8927108853657487, 'gamma': 3.7946171538376783, 'reg_alpha': 0.018155069209867845, 'reg_lambda': 0.021785298629927417}


[I 2026-09-21 16:16:34,696] Trial 0 finished with value: 0.6719321629450314 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.49359671220172163, 'gamma': 0.2904180608409973, 'reg_alpha': 1.1752647960576206, 'reg_lambda': 0.6358358856676253}. Best is trial 0 with value: 0.6719321629450314.
[I 2026-09-21 16:16:38,800] Trial 1 finished with value: 0.6531506160589203 and parameters: {'n_estimators': 450, 'max_depth': 2, 'learning_rate': 0.2708160864249968, 'min_child_weight': 9, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'gamma': 0.9170225492671691, 'reg_alpha': 0.0026892128247368906, 'reg_lambda': 0.3752055855124282}. Best is trial 0 with value: 0.6719321629450314.
[I 2026-09-21 16:16:42,721] Trial 2 finished with value: 0.6844135925874208 and parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.08012737503998542, 'min_child_

ES | F1-Macro=0.699641
{'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.06147854456275125, 'min_child_weight': 2, 'subsample': 0.8689001939048056, 'colsample_bytree': 0.40158494903420716, 'gamma': 4.927235313372161, 'reg_alpha': 0.10115112375831407, 'reg_lambda': 0.03316725102868027}


[I 2026-09-21 16:20:09,399] Trial 0 finished with value: 0.6051133749683603 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.49359671220172163, 'gamma': 0.2904180608409973, 'reg_alpha': 1.1752647960576206, 'reg_lambda': 0.6358358856676253}. Best is trial 0 with value: 0.6051133749683603.
[I 2026-09-21 16:20:13,631] Trial 1 finished with value: 0.6013256765181734 and parameters: {'n_estimators': 450, 'max_depth': 2, 'learning_rate': 0.2708160864249968, 'min_child_weight': 9, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'gamma': 0.9170225492671691, 'reg_alpha': 0.0026892128247368906, 'reg_lambda': 0.3752055855124282}. Best is trial 0 with value: 0.6051133749683603.
[I 2026-09-21 16:20:17,810] Trial 2 finished with value: 0.641149408607129 and parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.08012737503998542, 'min_child_w

CU | F1-Macro=0.665341
{'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.018041455877536097, 'min_child_weight': 2, 'subsample': 0.8331663580354526, 'colsample_bytree': 0.4970565102038133, 'gamma': 4.976934981139605, 'reg_alpha': 0.43874101810124466, 'reg_lambda': 0.07770219339788582}


In [33]:
comparacion_final = pd.DataFrame([
    {
        "variante": "mx",
        "optuna_v1": study_mx.best_value,
        "optuna_final": study_mx_final.best_value,
        "diferencia": (
            study_mx_final.best_value
            - study_mx.best_value
        )
    },
    {
        "variante": "es",
        "optuna_v1": study_es.best_value,
        "optuna_final": study_es_final.best_value,
        "diferencia": (
            study_es_final.best_value
            - study_es.best_value
        )
    },
    {
        "variante": "cu",
        "optuna_v1": study_cu.best_value,
        "optuna_final": study_cu_final.best_value,
        "diferencia": (
            study_cu_final.best_value
            - study_cu.best_value
        )
    }
])

comparacion_final

,variante,optuna_v1,optuna_final,diferencia
0,mx,0.623517,0.625833,0.002316
1,es,0.703954,0.699641,-0.004313
2,cu,0.666562,0.665341,-0.001222


In [34]:
STUDY_GANADOR = {
    "mx": study_mx_final,
    "es": study_es_final,
    "cu": study_cu
}

In [35]:
from sklearn.model_selection import GridSearchCV

def grid_local_variante(variante):

    df_train = cargar_train_ganador(variante)

    X = df_train[
        ["MESSAGE_CLEAN"] + FEATURE_COLS
    ]

    y = df_train["IS_IRONIC"].values

    scale_pos_weight = (
        (y == 0).sum() /
        (y == 1).sum()
    )

    # Mejor configuración encontrada con Optuna
    best = STUDY_GANADOR[variante].best_params.copy()

    # En CU puede venir guardado ngram_range
    best.pop("ngram_range", None)

    model = XGBClassifier(
        subsample=best["subsample"],
        colsample_bytree=best["colsample_bytree"],
        gamma=best["gamma"],
        reg_alpha=best["reg_alpha"],
        reg_lambda=best["reg_lambda"],

        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline([
        (
            "prep",
            build_tfidf_prep((1, 2))
        ),
        ("clf", model)
    ])

    # Grid pequeño alrededor de Optuna
    n_est = best["n_estimators"]
    depth = best["max_depth"]
    lr = best["learning_rate"]
    child = best["min_child_weight"]

    param_grid = {
        "clf__n_estimators": sorted(set([
            max(50, n_est - 50),
            n_est,
            n_est + 50
        ])),

        "clf__max_depth": sorted(set([
            max(2, depth - 1),
            depth,
            depth + 1
        ])),

        "clf__learning_rate": sorted(set([
            max(0.01, lr * 0.75),
            lr,
            min(0.30, lr * 1.25)
        ])),

        "clf__min_child_weight": sorted(set([
            max(1, child - 1),
            child,
            child + 1
        ]))
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=CV,
        n_jobs=1,
        verbose=1,
        refit=True
    )

    grid.fit(X, y)

    print(f"\n{variante.upper()}")
    print(
        "Optuna:",
        STUDY_GANADOR[variante].best_value
    )
    print(
        "GridSearch:",
        grid.best_score_
    )
    print(
        "Mejores parámetros:",
        grid.best_params_
    )

    return grid

In [36]:
grid_mx = grid_local_variante("mx")
grid_es = grid_local_variante("es")
grid_cu = grid_local_variante("cu")

Fitting 5 folds for each of 81 candidates, totalling 405 fits

MX
Optuna: 0.6258329468316003
GridSearch: 0.6258329468316003
Mejores parámetros: {'clf__learning_rate': 0.06299349502760707, 'clf__max_depth': 10, 'clf__min_child_weight': 3, 'clf__n_estimators': 300}
Fitting 5 folds for each of 81 candidates, totalling 405 fits

ES
Optuna: 0.6996407075532447
GridSearch: 0.7017270033413545
Mejores parámetros: {'clf__learning_rate': 0.06147854456275125, 'clf__max_depth': 4, 'clf__min_child_weight': 1, 'clf__n_estimators': 300}
Fitting 5 folds for each of 81 candidates, totalling 405 fits

CU
Optuna: 0.6665622610215698
GridSearch: 0.6616548378659992
Mejores parámetros: {'clf__learning_rate': 0.015890532329436432, 'clf__max_depth': 3, 'clf__min_child_weight': 2, 'clf__n_estimators': 650}


In [37]:
comparacion_grid = pd.DataFrame([
    {
        "variante": "mx",
        "optuna": STUDY_GANADOR["mx"].best_value,
        "grid": grid_mx.best_score_,
        "diferencia": (
            grid_mx.best_score_
            - STUDY_GANADOR["mx"].best_value
        )
    },
    {
        "variante": "es",
        "optuna": STUDY_GANADOR["es"].best_value,
        "grid": grid_es.best_score_,
        "diferencia": (
            grid_es.best_score_
            - STUDY_GANADOR["es"].best_value
        )
    },
    {
        "variante": "cu",
        "optuna": STUDY_GANADOR["cu"].best_value,
        "grid": grid_cu.best_score_,
        "diferencia": (
            grid_cu.best_score_
            - STUDY_GANADOR["cu"].best_value
        )
    }
])

comparacion_grid

,variante,optuna,grid,diferencia
0,mx,0.625833,0.625833,0.000000
1,es,0.699641,0.701727,0.002086
2,cu,0.666562,0.661655,-0.004907


In [38]:
CONFIG_FINAL_XGB = {
    "mx": {
        "source": "grid",
        "best_score_cv": grid_mx.best_score_,
        "best_params": grid_mx.best_params_
    },

    "es": {
        "source": "grid",
        "best_score_cv": grid_es.best_score_,
        "best_params": grid_es.best_params_
    },

    "cu": {
        "source": "grid",
        "best_score_cv": grid_cu.best_score_,
        "best_params": grid_cu.best_params_
    }
}

In [39]:
resumen_final_xgb = pd.DataFrame([
    {
        "variante": "mx",
        "metodo_seleccion": "GridSearch local",
        "f1_macro_cv": grid_mx.best_score_
    },
    {
        "variante": "es",
        "metodo_seleccion": "GridSearch local",
        "f1_macro_cv": grid_es.best_score_
    },
    {
        "variante": "cu",
        "metodo_seleccion": "GridSearch local",
        "f1_macro_cv": grid_cu.best_score_
    }
])

resumen_final_xgb

,variante,metodo_seleccion,f1_macro_cv
0,mx,GridSearch local,0.625833
1,es,GridSearch local,0.701727
2,cu,GridSearch local,0.661655
